<a href="https://colab.research.google.com/github/harrisonritz/DynamicsTutorial_CCN2026/blob/main/01_kalman_filter_information_form.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CCN Tutorial · Notebook 1
## Bayesian filtering in linear–Gaussian state-space models: the **Kalman filter** (information form)

State-space models (SSMs) describe a **latent** trajectory $x_{1:T}$ that we never observe directly, together with **noisy observations** $y_{1:T}$ that we do. Two questions define almost everything we do with them:

1. **Inference** — given the model, what is the latent state? This is *filtering / smoothing*. **(this notebook)**
2. **Learning** — given only $y_{1:T}$, what is the model? This is *parameter estimation*, usually via EM. **(lecture + Notebook 2)**

For the **linear–Gaussian** SSM both questions have exact, closed-form answers. Here we focus on question 1 and build the **Kalman filter**, written in its **information (canonical) form** — the parameterization that is cheapest exactly in the regime neuroscience lives in: *many* observed channels, *few* latent dimensions.

Everything runs on one worked example: a damped 2-D latent oscillator read out by 10 noisy channels.

**By the end you will be able to:**
- write down the linear–Gaussian SSM and the filtering recursion;
- explain the **predict** and **update** steps as operations on a Gaussian belief;
- contrast the **moment** and **information** parameterizations and say *when* each is cheaper;
- run the information filter and check that it matches `dynamax`'s `lgssm_info_filter` to machine precision;
- handle **missing data** — a dropped time point, or any subset of channels at any time point — as a masked sum, and read off how the posterior degrades as channels disappear.

We defer parameter learning (EM) to the lecture and Notebook 2.


## 0. Setup

We enable 64-bit precision in JAX (SSM inference involves matrix inverses; float32 is often too coarse for a clean comparison) and install `dynamax`, which we use later as an independent reference implementation.

In [ ]:
# On Colab this installs dynamax the first time; locally it is a no-op.
try:
    import dynamax  # noqa: F401
except ImportError:
    import subprocess, sys

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "dynamax"], check=True
    )

import numpy as np
import jax

jax.config.update("jax_enable_x64", True)  # match NumPy's float64
import jax.numpy as jnp
import matplotlib.pyplot as plt

plt.rcParams.update(
    {
        "figure.dpi": 300,
        "font.size": 11,
        "axes.grid": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.alpha": 0.25,
    }
)
rng = np.random.default_rng(0)

## 1. The linear–Gaussian state-space model

A latent state $x_t \in \mathbb{R}^{d_x}$ evolves as a noisy **linear** map; each observation $y_t \in \mathbb{R}^{d_y}$ is a noisy **linear** readout of the current state:

$$
\begin{aligned}
x_0 &\sim \mathcal{N}(m_0,\ V_0), &&&\text{(initial)}\\
x_t &= A\,x_{t-1} + w_t, & w_t &&\sim \mathcal{N}(0,\ Q), &&&\text{(dynamics)}\\
y_t &= C\,x_t + v_t, & v_t &&\sim \mathcal{N}(0,\ R). &&&\text{(observation)}
\end{aligned}
$$

The parameters are $\theta = (A, Q, C, R, m_0, V_0)$.
- $A$ is the **dynamics** matrix
- $C$ the **emission / loading** matrix
- $Q,R$ are process and observation noise covariances. 
- $Bu_t$, $Du_t$ are the inputs

Because the model is linear and every noise source is Gaussian, **every conditional distribution over states is Gaussian**. That single fact is what makes exact filtering possible: we only ever track a *mean* and a *covariance* (equivalently, an information vector and an information matrix), never a general density.

A note on vocabulary. The **filtering** distribution is $p(x_t \mid y_{1:t})$ — the belief about the state *now*, using data *up to now* (causal, online). The **smoothing** distribution $p(x_t \mid y_{1:T})$ conditions on the *whole* recording. This notebook is entirely about **filtering**.

### 1.1 A concrete example: a damped latent oscillator

We use a deliberately small, visualizable system: a **2-D latent** whose dynamics matrix $A$ is a rotation scaled by a factor $<1$, so trajectories **spiral inward** (a damped oscillation). We then project the 2-D latent up to a **10-D** noisy observation through a fixed random loading matrix $C$ — mimicking the common situation of a low-dimensional latent process read out by many noisy channels.

The behavior of a linear system is governed by the **eigenvalues** of $A$. Writing $\lambda = \rho e^{i\phi}$:
- $\rho = |\lambda|$ controls **stability**: $\rho<1 \Rightarrow$ decay, $\rho=1 \Rightarrow$ sustained, $\rho>1 \Rightarrow$ growth;
- $\phi = \arg(\lambda)$ sets the **oscillation frequency** (radians per step).

Here $A$ has complex-conjugate eigenvalues with $|\lambda| = 0.96$ (gentle decay) and a period of ~25 steps.

In [ ]:
dx, dy = 2, 10  # latent and observed dimensions

theta = 2 * np.pi / 25  # rotation: ~25-step period
radius = 0.96  # |eigenvalue| < 1  -> damped
A = radius * np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
Q = 0.05 * np.eye(dx)  # process noise

C = rng.normal(size=(dy, dx)) / np.sqrt(dx)  # random 2D -> 10D loading
sigma_obs = 0.5
R = (sigma_obs**2) * np.eye(dy)  # observation noise (diagonal)

m_0, V_0 = np.array([1.5, 0.0]), np.eye(dx)

eig = np.linalg.eigvals(A)
evec = np.linalg.eig(A)[1]
print("eigenvectors of A':\n", np.round(evec, 3))
print("eigenvalues of A:", np.round(eig, 3))
print("V'V - VV'=", np.round((evec.conj().T @ evec) - (evec @ evec.conj().T), 3))
print(
    "|lambda| =",
    np.round(np.abs(eig), 3),
    " period (steps) =",
    np.round(2 * np.pi / np.abs(np.angle(eig)), 2),
)

In [ ]:
# Simulate one trajectory from the generative model (fully transparent).
T = 200
x = np.zeros((T, dx))
y = np.zeros((T, dy))
x[0] = rng.multivariate_normal(m_0, V_0)
y[0] = C @ x[0] + rng.multivariate_normal(np.zeros(dy), R)
for t in range(1, T):
    x[t] = A @ x[t - 1] + rng.multivariate_normal(np.zeros(dx), Q)
    y[t] = C @ x[t] + rng.multivariate_normal(np.zeros(dy), R)

In [ ]:
# Figure 1a: the parameters themselves, as heat maps.
def show_mat(
    ax, M, title, xticklabels=None, yticklabels=None, annot=False, aspect="equal"
):
    # One parameter matrix, on its own color scale (magnitudes differ by ~20x, so a
    # shared scale would flatten Q to nothing). Signed matrices get a diverging map
    # centered at zero; non-negative ones a sequential map anchored at zero.
    M = np.atleast_2d(M)
    if M.min() < 0:
        v = np.abs(M).max()
        im = ax.imshow(M, cmap="RdBu_r", vmin=-v, vmax=v, aspect=aspect)
    else:
        im = ax.imshow(
            M, cmap="viridis", vmin=0, vmax=max(M.max(), 1e-12), aspect=aspect
        )
    ax.set_title(title, fontsize=10, pad=6)
    ax.grid(False)
    for lbls, axis, n in (
        (xticklabels, "x", M.shape[1]),
        (yticklabels, "y", M.shape[0]),
    ):
        set_ticks, set_lbls = (
            (ax.set_xticks, ax.set_xticklabels)
            if axis == "x"
            else (ax.set_yticks, ax.set_yticklabels)
        )
        if lbls is None:  # plain 1-based indices, thinned when there are many
            ticks = np.arange(n)[:: 2 if n > 6 else 1]
            set_ticks(ticks)
            set_lbls([str(i + 1) for i in ticks], fontsize=8)
        else:
            set_ticks(np.arange(n))
            set_lbls(lbls, fontsize=9)
    if annot:
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                r, g, b, _ = im.cmap(im.norm(M[i, j]))  # keep text readable on any cell
                col = "k" if 0.30 * r + 0.59 * g + 0.11 * b > 0.55 else "w"
                ax.text(
                    j,
                    i,
                    f"{M[i, j]:.2f}",
                    ha="center",
                    va="center",
                    color=col,
                    fontsize=9,
                )
    cb = ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.ax.tick_params(labelsize=8)
    return im


lat = ["$x_1$", "$x_2$"]
fig = plt.figure(figsize=(9.5, 6.0))
outer = fig.add_gridspec(2, 1, height_ratios=[1, 1.7], hspace=0.45)
top = outer[0].subgridspec(1, 4, wspace=0.75)
bot = outer[1].subgridspec(1, 2, width_ratios=[1, 2.6], wspace=0.05)
show_mat(
    fig.add_subplot(top[0]),
    m_0[:, None],
    r"$m_0$  (prior mean)",
    [""],
    lat,
    annot=True,
    aspect="auto",
)
show_mat(
    fig.add_subplot(top[1]),
    V_0,
    r"$ V_0$  (prior cov.)",
    lat,
    lat,
    annot=True,
    aspect="auto",
)
show_mat(
    fig.add_subplot(top[2]), A, r"$A$  (dynamics)", lat, lat, annot=True, aspect="auto"
)
show_mat(
    fig.add_subplot(top[3]),
    Q,
    r"$Q$  (process noise)",
    lat,
    lat,
    annot=True,
    aspect="auto",
)
show_mat(fig.add_subplot(bot[0]), C, r"$C$  (loadings, $10\times2$)", lat, None)
show_mat(fig.add_subplot(bot[1]), R, r"$R$  (obs. noise, $10\times10$; diagonal)")
fig.suptitle(
    r"Model parameters $\theta=(A,Q,C,R,m_0, V_0)$ — each panel on its own color scale",
    fontsize=12,
)
plt.show()

# Figure 1b: eigenvalues of A, and the latent phase portrait.
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
phi = np.linspace(0, 2 * np.pi, 200)
ax[0].plot(np.cos(phi), np.sin(phi), "k--", lw=1)
ax[0].scatter(eig.real, eig.imag, s=80, zorder=3, color="C3")
ax[0].axhline(0, color="0.7", lw=0.6)
ax[0].axvline(0, color="0.7", lw=0.6)
ax[0].set(
    title="Eigenvalues of $A$ (unit circle)",
    xlabel="Re",
    ylabel="Im",
    xlim=(-1.15, 1.15),
    ylim=(-1.15, 1.15),
    aspect="equal",
)

ax[1].plot(x[:, 0], x[:, 1], lw=1, color="C0")
ax[1].scatter(*x[0], color="C2", zorder=3, label="start")
ax[1].scatter(*x[-1], color="C3", zorder=3, label="end")
ax[1].set(title="Latent trajectory $x_t$", xlabel="$x_1$", ylabel="$x_2$")
ax[1].legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 2: what we OBSERVE (noisy, high-D) vs. what we WANT (clean, low-D).
fig, ax = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
for i in range(dy):
    ax[0].plot(y[:, i], lw=0.8, alpha=0.6)
ax[0].set(
    title=f"Observations $y_t$  ({dy} noisy channels) — what we measure", ylabel="$y$"
)
ax[1].plot(x[:, 0], label="$x_1$", color="C0")
ax[1].plot(x[:, 1], label="$x_2$", color="C1")
ax[1].set(
    title="Latent state $x_t$ (2-D) — what we want to recover",
    xlabel="time step $t$",
    ylabel="$x$",
)
ax[1].legend(frameon=False, ncol=2)
plt.tight_layout()
plt.show()

## 2. Filtering as a recursion

We want the filtering belief $p(x_t \mid y_{1:t})$ for every $t$, computed **online**. The Kalman filter obtains it by alternating two steps, each of which maps one Gaussian to another:

**Predict (time update)** — push the previous belief through the dynamics:
$$
p(x_t \mid y_{1:t-1}) \;=\; \int p(x_t \mid x_{t-1})\; p(x_{t-1}\mid y_{1:t-1})\, dx_{t-1}.
$$

**Update (measurement update)** — fold in the new observation via Bayes' rule:
$$
p(x_t \mid y_{1:t}) \;\propto\; p(y_t \mid x_t)\; p(x_t \mid y_{1:t-1}).
$$

Both operations preserve Gaussianity, so the filter just transports the *parameters* of a Gaussian. The only real choice is **which parameters**.

### 2.1 Moment form (the familiar Kalman filter)

Track the mean $m_{t\mid t}$ and covariance $V_{t\mid t}$. With $\hat m = m_{t\mid t-1}$, $\hat V = V_{t\mid t-1}$:

$$
\textbf{predict:}\quad m_{t\mid t-1} = A\,m_{t-1\mid t-1}, \qquad V_{t\mid t-1} = A\,V_{t-1\mid t-1}\,A^\top + Q,
$$

$$
\textbf{update:}\quad
S_t = C\hat V C^\top + R,\quad
K_t = \hat V C^\top S_t^{-1},\quad
m_{t\mid t} = \hat m + K_t(y_t - C\hat m),\quad
V_{t\mid t} = (I - K_t C)\hat V .
$$

The **innovation** $y_t - C\hat m$ is the surprise in the observation; the **Kalman gain** $K_t$ says how much to trust it. Note the update inverts $S_t$, a $d_y \times d_y$ matrix — its cost grows with the number of **observation** channels.

### 2.2 Information form (the focus of this notebook)

A Gaussian can be written in **canonical / information** parameters instead of moments. Starting from
$$
\mathcal{N}(x;m, V)\ \propto\ \exp\!\Big(-\tfrac12\,x^\top  V^{-1} x + x^\top  V^{-1}m\Big),
$$
define the **information matrix** (precision) and **information vector**
$$
\Lambda =  V^{-1}, \qquad \eta = \Lambda\,m,
\qquad\text{so}\qquad
\mathcal{N}(x)\ \propto\ \exp\!\Big(-\tfrac12\,x^\top \Lambda\,x + x^\top \eta\Big).
$$

**Update becomes pure addition.** Conditioning on $y_t = C x_t + v_t$, $v_t\sim\mathcal N(0,R)$:
$$
\boxed{\;\Lambda_{t\mid t} = \Lambda_{t\mid t-1} + C^\top R^{-1} C,
\qquad
\eta_{t\mid t} = \eta_{t\mid t-1} + C^\top R^{-1} y_t\;}
$$
Each observation simply **adds** the information $C^\top R^{-1}C$ (and $C^\top R^{-1}y_t$ to the information vector). There is **no $d_y\times d_y$ inverse** — only $R^{-1}$, which for independent channels is trivial. Three consequences fall straight out:
- **Conditionally independent observations just sum:** $\sum_i C_i^\top R_{ii}^{-1} C_i$. Sensor fusion is addition.
- **Missing data is free:** if $y_t$ is unobserved, add nothing — $\Lambda_{t\mid t}=\Lambda_{t\mid t-1}$.
- **A diffuse (uninformative) prior is representable:** $\Lambda_0 = 0$ means "infinite uncertainty," which the moment form ($V_0 = \infty$) cannot express.

**Predict is now the harder step.** Marginalizing the dynamics gives (via the matrix-inversion lemma)
$$
\Lambda_{t+1\mid t} = Q^{-1} - Q^{-1}A\big(\Lambda_{t\mid t} + A^\top Q^{-1} A\big)^{-1} A^\top Q^{-1}.
$$
`dynamax` computes an algebraically equivalent, guaranteed-symmetric form that we mirror exactly below:
$$
K = Q^{-1}A\big(\Lambda_{t\mid t} + A^\top Q^{-1} A\big)^{-1},\qquad
L = I - K A^\top,
$$
$$
\Lambda_{t+1\mid t} = L\,Q^{-1}L^\top + K\,\Lambda_{t\mid t}\,K^\top,
\qquad
\eta_{t+1\mid t} = K\,\eta_{t\mid t}.
$$
The only inverse here is the $d_x\times d_x$ solve $(\Lambda_{t\mid t}+A^\top Q^{-1}A)^{-1}$ — its cost grows with the number of **latent** dimensions.

**The duality, in one line.** Update is cheap in information form (no $d_y$ inverse); predict is cheap in moment form (no inverse at all). When $d_y \gg d_x$ — *many channels, few latents*, the typical neural-data regime — the information filter wins, because you pay in the small dimension $d_x$ and save in the large dimension $d_y$.

In [ ]:
def information_filter(y, A, Q, C, R, m_0, V_0, observed=None):
    # Kalman filter in information form. Returns filtered (eta, Lambda) per step.
    # observed marks which data are present, and may be either
    #   shape (T,)     -- observed[t] == False drops the whole observation, or
    #   shape (T, dy)  -- observed[t, i] == False drops only channel i at time t.
    # Assumes R is diagonal, so dropping a channel means dropping its term in the sum.
    T, dy = y.shape
    dx = A.shape[0]
    Qi, Ri = np.linalg.inv(Q), np.linalg.inv(R)
    CtRi = C.T @ Ri
    chan_info = np.einsum("ij,i,ik->ijk", C, np.diag(Ri), C)  # C_i' R_ii^-1 C_i, per i
    I = np.eye(dx)

    observed = np.ones((T, dy)) if observed is None else np.asarray(observed, float)
    if observed.ndim == 1:  # broadcast an all-or-nothing mask over channels
        observed = observed[:, None] * np.ones(dy)

    etas = np.zeros((T, dx))
    Lams = np.zeros((T, dx, dx))
    eta_pred, Lam_pred = np.linalg.inv(V_0) @ m_0, np.linalg.inv(V_0)  # t=0 prior

    for t in range(T):
        # --- measurement update (ADDITIVE: add only the observed channels) ---
        m = observed[t]
        eta = eta_pred + CtRi @ (m * y[t])
        Lam = Lam_pred + np.einsum("i,ijk->jk", m, chan_info)
        etas[t], Lams[t] = eta, Lam

        # --- time update / predict (one dx-by-dx solve) ---
        K = Qi @ A @ np.linalg.inv(Lam + A.T @ Qi @ A)
        L = I - K @ A.T
        Lam_pred = L @ Qi @ L.T + K @ Lam @ K.T
        eta_pred = K @ eta
    return etas, Lams


def to_moments(etas, Lams):
    mu = np.stack([np.linalg.solve(Lams[t], etas[t]) for t in range(len(etas))])
    V = np.stack([np.linalg.inv(Lams[t]) for t in range(len(etas))])
    return mu, V

In [ ]:
etas, Lams = information_filter(y, A, Q, C, R, m_0, V_0)
mu_filt, V_filt = to_moments(etas, Lams)
print(
    "filtered-vs-true latent RMSE:",
    round(float(np.sqrt(((mu_filt - x) ** 2).mean())), 4),
)

### 2.3 Cross-check against `dynamax`

`dynamax` ships an information filter, `lgssm_info_filter`, parameterized directly by **precisions**: `dynamics_precision` $=Q^{-1}$, `emission_precision` $=R^{-1}$, `initial_precision` $= V_0^{-1}$. We run it on the same data and confirm our hand-written filter agrees to machine precision — and, for good measure, that both agree with the standard moment-form filter `lgssm_filter`.

In [ ]:
from dynamax.linear_gaussian_ssm import (
    LinearGaussianSSM,
    ParamsLGSSMInfo,
    lgssm_info_filter,
)
from dynamax.linear_gaussian_ssm.info_inference import info_to_moment_form

seed = 99

info_params = ParamsLGSSMInfo(
    initial_mean=jnp.array(m_0),
    initial_precision=jnp.linalg.inv(jnp.array(V_0)),
    dynamics_weights=jnp.array(A),
    dynamics_precision=jnp.linalg.inv(jnp.array(Q)),
    dynamics_input_weights=jnp.zeros((dx, 0)),
    dynamics_bias=jnp.zeros(dx),
    emission_weights=jnp.array(C),
    emission_precision=jnp.linalg.inv(jnp.array(R)),
    emission_input_weights=jnp.zeros((dy, 0)),
    emission_bias=jnp.zeros(dy),
)

post_info = lgssm_info_filter(info_params, jnp.array(y))
mu_dmax, V_dmax = info_to_moment_form(
    post_info.filtered_etas, post_info.filtered_precisions
)
mu_dmax, V_dmax = np.array(mu_dmax), np.array(V_dmax)

# standard moment-form filter, same parameters
model = LinearGaussianSSM(state_dim=dx, emission_dim=dy)
params, _ = model.initialize(
    jax.random.key(seed),
    initial_mean=jnp.array(m_0),
    initial_covariance=jnp.array(V_0),
    dynamics_weights=jnp.array(A),
    dynamics_covariance=jnp.array(Q),
    emission_weights=jnp.array(C),
    emission_covariance=jnp.array(R),
)
post_mom = model.filter(params, jnp.array(y))

print(
    "max |mean| difference, ours vs dynamax info-filter:  "
    f"{np.abs(mu_filt - mu_dmax).max():.2e}"
)
print(
    "max |cov|  difference, ours vs dynamax info-filter:  "
    f"{np.abs(V_filt - V_dmax).max():.2e}"
)
print(
    "max |mean| difference, info-form vs moment-form:     "
    f"{np.abs(mu_dmax - np.array(post_mom.filtered_means)).max():.2e}"
)
print(
    "marginal log-likelihood (from dynamax):             "
    f"{float(post_info.marginal_loglik):.3f}"
)

In [ ]:
# Figure 3: the filter recovers the latent state it never directly saw.
fig, ax = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
for i, a in enumerate(ax):
    sd = np.sqrt(V_filt[:, i, i])
    a.plot(x[:, i], color="k", lw=1.5, label="true $x$")
    a.plot(mu_filt[:, i], color="C3", lw=1.5, label="filtered mean")
    a.fill_between(
        np.arange(T),
        mu_filt[:, i] - 2 * sd,
        mu_filt[:, i] + 2 * sd,
        color="C3",
        alpha=0.2,
        label=r"$\pm 2 V$",
    )
    a.set_ylabel(f"$x_{i + 1}$")
ax[0].set_title("Filtered estimate vs. ground truth")
ax[0].legend(frameon=False, ncol=3, loc="upper right")
ax[-1].set_xlabel("time step $t$")
plt.tight_layout()
plt.show()

# Missing data

**All-or-nothing gaps.** If an observation is absent, we simply skip the additive update — the belief coasts forward on the dynamics alone and its uncertainty grows, then contracts once data returns. This much is not special to the information form: *any* Bayesian filter handles a missing observation by setting the Kalman gain to zero ($m_{t\mid t}=m_{t\mid t-1}$, $V_{t\mid t}=V_{t\mid t-1}$). Below we blank out steps 80–110 and track the total posterior uncertainty $\sqrt{\operatorname{tr} V_{t\mid t}}$.

In [ ]:
observed = np.ones(T, dtype=bool)
observed[80:110] = False  # a gap with no observations

etas_g, Lams_g = information_filter(y, A, Q, C, R, m_0, V_0, observed=observed)
mu_g, V_g = to_moments(etas_g, Lams_g)
tot_sd = np.sqrt(np.trace(V_g, axis1=1, axis2=2))

fig, ax = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
sd0 = np.sqrt(V_g[:, 0, 0])
ax[0].plot(x[:, 0], "k", lw=1.5, label="true $x_1$")
ax[0].plot(mu_g[:, 0], "C3", lw=1.5, label="filtered mean")
ax[0].fill_between(
    np.arange(T), mu_g[:, 0] - 2 * sd0, mu_g[:, 0] + 2 * sd0, color="C3", alpha=0.2
)
ax[0].axvspan(80, 110, color="C7", alpha=0.18, label="no observations")
ax[0].set_ylabel("$x_1$")
ax[0].legend(frameon=False, ncol=3, loc="upper right")
ax[1].plot(tot_sd, color="C0")
ax[1].axvspan(80, 110, color="C7", alpha=0.18)
ax[1].set(
    ylabel=r"$\sqrt{\mathrm{tr}\,V_{t|t}}$",
    xlabel="time step $t$",
    title="Posterior uncertainty grows without data, then contracts",
)
plt.tight_layout()
plt.show()

**Partial observations — where the information form actually earns its keep.** The realistic case is not that *all* channels vanish, but that *some* do: a dropped electrode, a censored fMRI volume, a subject who moved. Let $\mathcal{O}_t \subseteq \{1,\dots,d_y\}$ be the channels available at time $t$. Because the update is a sum over channels, we just leave out the missing terms:

$$
\Lambda_{t\mid t} = \Lambda_{t\mid t-1} + \sum_{i \in \mathcal{O}_t} C_{i,:}^\top R_{ii}^{-1} C_{i,:},
\qquad
\eta_{t\mid t} = \eta_{t\mid t-1} + \sum_{i \in \mathcal{O}_t} C_{i,:}^\top R_{ii}^{-1} y_{t,i}.
$$

The operand keeps the same $d_x \times d_x$ shape no matter which channels survive, so this is a masked sum — vectorizable, `jit`-able, no branching. The moment form gets the *same answer* (we check this below), but it has to subset the rows of $C$ and invert an innovation covariance $S_t$ whose **dimension changes from step to step** — awkward to write and impossible to batch cleanly.

The behavior this exposes is worth dwelling on: uncertainty **degrades gracefully**, interpolating between "all data" and "no data" as channels drop out.

In [ ]:
# A staircase of channel dropout: 10 -> 5 -> 2 -> 0 -> 10 channels observed.
blocks = [(0, 60, 10), (60, 90, 5), (90, 120, 2), (120, 140, 0), (140, T, 10)]
k_sched = np.empty(T, int)
for a, b, k in blocks:
    k_sched[a:b] = k

priority = np.random.default_rng(7).permutation(dy)  # which channels survive a cut
mask = np.zeros((T, dy))
for t in range(T):
    mask[t, priority[: k_sched[t]]] = 1.0

etas_p, Lams_p = information_filter(y, A, Q, C, R, m_0, V_0, observed=mask)
mu_p, V_p = to_moments(etas_p, Lams_p)
sd_p = np.sqrt(np.trace(V_p, axis1=1, axis2=2))
sd_full = np.sqrt(np.trace(V_filt, axis1=1, axis2=2))  # all 10 channels, from above

fig, ax = plt.subplots(
    3, 1, figsize=(9, 7), sharex=True, gridspec_kw={"height_ratios": [0.6, 1, 1]}
)
ax[0].imshow(
    mask.T, aspect="auto", cmap="Greys", vmin=0, vmax=1.7, interpolation="nearest"
)
ax[0].set(
    ylabel="channel",
    yticks=[0, 5, 9],
    title=r"Observation mask (shaded = observed); numbers give $|\mathcal{O}_t|$",
)
ax[0].grid(False)
for a, b, k in blocks:
    ax[0].text(
        (a + b) / 2, -0.35, f"{k}", ha="center", va="top", color="C3", fontweight="bold"
    )

sd0 = np.sqrt(V_p[:, 0, 0])
ax[1].plot(x[:, 0], "k", lw=1.5, label="true $x_1$")
ax[1].plot(mu_p[:, 0], "C3", lw=1.5, label="filtered mean")
ax[1].fill_between(
    np.arange(T),
    mu_p[:, 0] - 2 * sd0,
    mu_p[:, 0] + 2 * sd0,
    color="C3",
    alpha=0.2,
    label=r"$\pm 2 V$",
)
ax[1].set_ylabel("$x_1$")
ax[1].legend(frameon=False, ncol=3, loc="upper right")

ax[2].plot(sd_p, color="C0", lw=1.5, label="partial observations")
ax[2].plot(sd_full, color="0.5", lw=1.2, ls="--", label="all 10 channels")
ax[2].axhline(sd_p[139], color="C7", lw=1, ls=":", label="no-data level")
ax[2].set(
    ylabel=r"$\sqrt{\mathrm{tr}\,V_{t|t}}$",
    xlabel="time step $t$",
    title="Uncertainty degrades gracefully with the number of channels",
)
ax[2].legend(frameon=False, ncol=3, loc="upper left")
for a, b, k in blocks[1:]:
    for a_ in ax:
        a_.axvline(a, color="0.6", lw=0.6, ls=":")
plt.tight_layout()
plt.show()

Notice the steps are **not** evenly spaced: dropping from 10 channels to 5 barely moves the posterior, while dropping from 2 to 0 is catastrophic. That is precision adding *linearly* while the standard deviation goes as its inverse square root — the first channel buys most of the certainty, and each additional one buys less. Two further consequences, both visible below:

- **It is not just the count, it is the geometry.** Each channel contributes $C_{i,:}^\top R_{ii}^{-1} C_{i,:}$, a rank-1 matrix pointing along $C_{i,:}$. A channel that loads on a latent direction you already measure well adds almost nothing; a channel spanning the *missing* direction is worth several redundant ones. Hence the spread across random channel subsets at fixed $|\mathcal{O}|$.
- **Uncertainty plateaus rather than diverging.** With $|\lambda(A)| = 0.96 < 1$ the process is stable, so with no data at all the belief relaxes to the model's stationary distribution — the flat ceiling in the previous figure. For a random walk ($|\lambda| = 1$) it would grow without bound.

We also confirm the masked information update against a moment-form filter that literally subsets the rows of $C$ and inverts a differently sized $S_t$ at each step. They agree to machine precision — the information form is not doing anything the moment form cannot, it is doing the same thing without the reshaping.

In [ ]:
# Steady-state uncertainty as a function of how many channels are observed.
rng_sub = np.random.default_rng(1)
n_rep = 40
ss = np.zeros((dy + 1, n_rep))
for k in range(dy + 1):
    for r in range(n_rep):
        m = np.zeros((T, dy))
        m[:, rng_sub.permutation(dy)[:k]] = 1.0  # a random subset of k channels
        _, Lam_k = information_filter(y, A, Q, C, R, m_0, V_0, observed=m)
        _, V_k = to_moments(_, Lam_k)
        ss[k, r] = np.sqrt(np.trace(V_k, axis1=1, axis2=2))[-50:].mean()

ks = np.arange(dy + 1)
fig, ax = plt.subplots(figsize=(9, 3.6))
for k in ks:
    ax.scatter(np.full(n_rep, k), ss[k], s=12, color="C0", alpha=0.25, lw=0)
ax.plot(
    ks, ss.mean(1), "o-", color="C3", lw=1.5, ms=5, label="mean over random subsets"
)
ax.set(
    xlabel=r"number of observed channels $|\mathcal{O}|$",
    ylabel=r"steady-state $\sqrt{\mathrm{tr}\,V}$",
    xticks=ks,
    title="Precision adds linearly, so the first channel buys the most",
)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


# --- cross-check: moment form with the same per-channel mask ---
def moment_filter_masked(y, A, Q, C, R, m_0, V_0, mask):
    # The same computation in moment form: subset the rows of C and invert an
    # innovation covariance whose size changes with |O_t|.
    T, dy = y.shape
    mu, V, I = m_0.copy(), V_0.copy(), np.eye(A.shape[0])
    mus, Ps = np.zeros((T, len(m_0))), np.zeros((T, len(m_0), len(m_0)))
    for t in range(T):
        if t > 0:
            mu, V = A @ mu, A @ V @ A.T + Q
        idx = np.flatnonzero(mask[t])
        if idx.size:
            Co, Ro = C[idx], R[np.ix_(idx, idx)]
            S = Co @ V @ Co.T + Ro  # <- idx.size x idx.size, a different size each t
            K = V @ Co.T @ np.linalg.inv(S)
            mu = mu + K @ (y[t][idx] - Co @ mu)
            V = (I - K @ Co) @ V
        mus[t], Ps[t] = mu, V
    return mus, Ps


mu_m, V_m = moment_filter_masked(y, A, Q, C, R, m_0, V_0, mask)
print(f"info vs moment form, max |mean| difference: {np.abs(mu_m - mu_p).max():.2e}")
print(f"info vs moment form, max |cov|  difference: {np.abs(V_m - V_p).max():.2e}")

## Recap & what's next

- The **linear–Gaussian SSM** keeps every state belief Gaussian, so filtering is exact: alternate **predict** (push through the dynamics) and **update** (fold in the observation).
- The **information form** re-parameterizes that same belief by precision $\Lambda= V^{-1}$ and information vector $\eta=\Lambda m$. The **update becomes additive** — no observation-dimension inverse — which is exactly why it shines when $d_y \gg d_x$, and why missing data, sensor fusion, and diffuse priors are effortless. The cost moves to the **predict** step, paid in the small latent dimension $d_x$.
- Our implementation reproduces `dynamax`'s `lgssm_info_filter` to machine precision.

**Next.** Filtering assumed the parameters $\theta$ were *known*. In practice we learn them from data. The **E-step** of EM is exactly a Kalman **smoother** (filtering's non-causal sibling, using all of $y_{1:T}$); the **M-step** updates $\theta$ in closed form. **Notebook 2** builds the full simulate → fit → recover loop and confronts the identifiability of these models head-on. **Notebook 3** lets the dynamics themselves switch over time (a *switching* LDS) and applies it to resting-state fMRI.